In [ ]:
# Notebook used to check the 2011 Census OD matrix data

In [ ]:
import sqlite3
import pandas as pd

CRS = "EPSG:31370"  # Belgian Lambert 72

### People FROM zzzz

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT 
    SUM(CAST(OBS_VALUE AS INTEGER)) AS num_people
FROM 'VU_GEO_LPW_MATRIX'
WHERE CD_RGN_RESIDENCE = '04000'
AND CD_SECTOR_RESIDENCE LIKE '%ZZZ'
"""
num_people_from_zzz = pd.read_sql_query(query, conn)
conn.close()

print(num_people_from_zzz)

# CONCLUSION: There are only 131 from ZZZ sectors, so these can be ignored

### People TO FOR

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT 
    SUM(CAST(OBS_VALUE AS INTEGER)) AS num_people
FROM 'VU_GEO_LPW_MATRIX'
WHERE CD_RGN_RESIDENCE = '04000'
AND CD_SECTOR_WORK LIKE '%FOR'
"""
num_people_to_for = pd.read_sql_query(query, conn)
conn.close()

print(num_people_to_for)

# CONCLUSION: There are only 2273 to FOR, so these can be ignored (total number of agents around 370k)

### People TO zzz

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT 
    SUM(CAST(OBS_VALUE AS INTEGER)) AS num_people
FROM 'VU_GEO_LPW_MATRIX'
WHERE CD_RGN_RESIDENCE = '04000'
AND CD_SECTOR_RESIDENCE NOT LIKE '%ZZZZ'
AND CD_SECTOR_WORK NOT LIKE '%FOR'
AND CD_SECTOR_WORK LIKE '%ZZZ'
"""
num_people_to_zzz = pd.read_sql_query(query, conn)
conn.close()

print(num_people_to_zzz)

# CONCLUSION: There are 43068 people to ZZZ, which is around 12% so these SHOULDN'T be ignored

### People to ZZZZZZ

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT 
    CD_SECTOR_WORK,
    SUM(CAST(OBS_VALUE AS INTEGER)) AS num_people
FROM 'VU_GEO_LPW_MATRIX'
WHERE CD_RGN_RESIDENCE = '04000'
AND CD_SECTOR_RESIDENCE NOT LIKE '%ZZZ'
AND CD_SECTOR_WORK NOT LIKE '%FOR'
AND CD_SECTOR_WORK LIKE '%ZZZZZ'
GROUP BY 1
"""
num_people_to_zzz = pd.read_sql_query(query, conn)
conn.close()

print(num_people_to_zzz)

# CONCLUSION: There are 14390 people to ZZZZZZ, but 14360 are to BE10Z_ZZZZZZZZZ, so rest can be ignored

### Statistical sectors stats

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT 
    COUNT(DISTINCT CD_SECTOR_RESIDENCE) AS num_home_sectors,
    COUNT(DISTINCT CD_SECTOR_WORK)      AS num_work_sectors,
    COUNT(DISTINCT CASE WHEN CD_RGN_WORK = '04000' THEN CD_SECTOR_WORK ELSE NULL END)      AS num_work_sectors_in_brussels
FROM 'VU_GEO_LPW_MATRIX'
WHERE CD_RGN_RESIDENCE = '04000'
AND CD_SECTOR_RESIDENCE NOT LIKE '%ZZZZ'
AND CD_SECTOR_WORK NOT LIKE '%FOR'
"""
num_stat_sectors = pd.read_sql_query(query, conn)
conn.close()

print(num_stat_sectors)

# CONCLUSION: 697 home stat sectors, 6170 work stat sectors, 727 in BCR

### People living and working in the same sector

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT 
    SUM(CAST(OBS_VALUE AS INTEGER)) AS total_num_people,
    SUM(CASE WHEN CD_SECTOR_RESIDENCE = CD_SECTOR_WORK THEN CAST(OBS_VALUE AS INTEGER)
        ELSE NULL END) AS people_in_same_sector
FROM 'VU_GEO_LPW_MATRIX'
WHERE CD_RGN_RESIDENCE = '04000'
"""
num_people_same_sector = pd.read_sql_query(query, conn)
conn.close()

print(num_people_same_sector)

# CONCLUSION: 370380 total people, 50174 living and working in the same sector --> 13.5%

### Sectors with highest same sector home-work flows

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the data
query = """
SELECT
    CD_SECTOR_RESIDENCE,
    people_in_same_sector/total_num_people * 100 AS pct_in_same_sector,
    total_num_people,
    people_in_same_sector
FROM (
    SELECT 
        CD_SECTOR_RESIDENCE,
        SUM(CAST(OBS_VALUE AS FLOAT)) AS total_num_people,
        SUM(CASE WHEN CD_SECTOR_RESIDENCE = CD_SECTOR_WORK THEN CAST(OBS_VALUE AS FLOAT)
            ELSE NULL END) AS people_in_same_sector
    FROM 'VU_GEO_LPW_MATRIX'
    WHERE CD_RGN_RESIDENCE = '04000'
    GROUP BY 1 
)
WHERE total_num_people > 100
ORDER BY pct_in_same_sector DESC
"""
num_people_same_sector = pd.read_sql_query(query, conn)
conn.close()

print(num_people_same_sector.head(10))